# 05 — Final Load Preparation | NYC Rolling Sales
> Data preparation for Tableau dashboarding and KPI analysis

## 1 · Load Cleaned Dataset

In [1]:
# -- 1. Imports and Configuration
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', '{:,.2f}'.format)

INPUT_PATH = os.path.join('..', 'data', 'processed', 'cleaned_data.csv')
OUTPUT_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
# -- 2. Load the cleaned dataset
df = pd.read_csv(INPUT_PATH, parse_dates=['sale_date'])
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range: {df['sale_date'].min().date()} to {df['sale_date'].max().date()}")
print(f"\nColumns: {list(df.columns)}")

Loaded dataset: 57,601 rows x 31 columns
Date range: 2016-09-01 to 2017-08-31

Columns: ['borough', 'borough_name', 'neighborhood', 'zip_code', 'building_class_category', 'building_class_code', 'building_class_description', 'building_class_at_present', 'building_class_at_time_of_sale', 'tax_class_at_present', 'tax_class_at_time_of_sale', 'residential_units', 'commercial_units', 'total_units', 'land_square_feet', 'gross_square_feet', 'year_built', 'sale_date', 'sale_price', 'sale_year', 'sale_month', 'sale_quarter', 'sale_day_of_week', 'price_per_sqft', 'price_per_unit', 'building_age', 'floor_area_ratio', 'is_residential', 'has_units', 'size_category', 'price_bracket']


In [3]:
# -- 3. Quick data integrity check
print("Null counts (non-zero only):")
nulls = df.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))
print(f"\nSale price range: ${df['sale_price'].min():,.0f} - ${df['sale_price'].max():,.0f}")
print(f"Median sale price: ${df['sale_price'].median():,.0f}")

Null counts (non-zero only):
floor_area_ratio             29650
gross_square_feet            29649
price_per_sqft               29649
size_category                29649
land_square_feet             28725
price_per_unit               16359
year_built                    4051
building_age                  4051
building_class_at_present      584
tax_class_at_present           584
dtype: int64

Sale price range: $25,000 - $14,000,000
Median sale price: $636,000


## 2 · Standardize Categoricals and Dates

Before computing anything, normalize all categorical text to Title Case and fill category-level nulls with explicit labels so Tableau filters show clean values instead of blank entries.

In [4]:
# -- 4. Standardize categorical columns
df['neighborhood'] = df['neighborhood'].str.strip().str.title()
df['building_class_description'] = df['building_class_description'].str.strip().str.title()
df['building_class_category'] = df['building_class_category'].str.strip().str.title()
df['sale_day_of_week'] = df['sale_day_of_week'].str.strip().str.title()

df['size_category'] = df['size_category'].fillna('Unknown')
df['price_bracket'] = df['price_bracket'].fillna('Unknown')

df['zip_code'] = df['zip_code'].astype(str).str.strip()
df['tax_class_at_time_of_sale'] = df['tax_class_at_time_of_sale'].astype(str).str.strip()
df['tax_class_at_present'] = df['tax_class_at_present'].astype(str).str.strip()

df['year_built'] = df['year_built'].astype('Int64')
df['building_age'] = df['building_age'].astype('Int64')

df['is_residential_flag'] = df['is_residential'].map({1: 'Residential', 0: 'Non-Residential'})

df['price_per_sqft'] = df['price_per_sqft'].round(2)
df['price_per_unit'] = df['price_per_unit'].round(2)
df['floor_area_ratio'] = df['floor_area_ratio'].round(3)

print("Categoricals standardized.")
print(f"Neighborhood sample: {df['neighborhood'].unique()[:5]}")
print(f"Property type sample: {df['building_class_description'].unique()[:5]}")
print(f"Size category values: {df['size_category'].unique()}")

Categoricals standardized.
Neighborhood sample: ['Alphabet City' 'Chelsea' 'Chinatown' 'Civic Center' 'Clinton']
Property type sample: ['Rentals - Walkup Apartments' 'Rentals - Elevator Apartments'
 'Coops - Walkup Apartments' 'Coops - Elevator Apartments' 'Condo-Rentals']
Size category values: ['Large' 'Mid-Large' 'Very Large' 'Unknown' 'Medium' 'Small']


In [5]:
# -- 5. Create Tableau-friendly date column (YYYY-MM-DD string)
df['sale_date_str'] = df['sale_date'].dt.strftime('%Y-%m-%d')

df['year_month'] = df['sale_date'].dt.to_period('M').astype(str)

df['month_start_date'] = df['sale_date'].dt.to_period('M').dt.start_time.dt.strftime('%Y-%m-%d')

print("Date columns created.")
print(f"sale_date_str sample: {df['sale_date_str'].iloc[0]}")
print(f"year_month sample: {df['year_month'].iloc[0]}")
print(f"month_start_date sample: {df['month_start_date'].iloc[0]}")

Date columns created.
sale_date_str sample: 2017-07-19
year_month sample: 2017-07
month_start_date sample: 2017-07-01


## 3 · Compute KPI Metrics

| KPI | Definition | Business Value |
|-----|-----------|----------------|
| Total Sales Volume | Sum of all sale prices | Overall market size and capital flow |
| Average Sale Price | Mean sale price per transaction | Market pricing benchmark |
| Total Transactions | Count of all sales | Market liquidity and activity level |
| Average Price per Sq Ft | Mean price per square foot | Normalized valuation across property sizes |

In [6]:
# -- 6. Compute top-level KPI summary
total_sales_volume = df['sale_price'].sum()
avg_sale_price = df['sale_price'].mean()
total_transactions = len(df)
avg_price_per_sqft = df['price_per_sqft'].dropna().mean()

kpi_summary = pd.DataFrame({
    'KPI': [
        'Total Sales Volume',
        'Average Sale Price',
        'Total Transactions',
        'Average Price Per SqFt'
    ],
    'Value': [
        round(total_sales_volume, 2),
        round(avg_sale_price, 2),
        int(total_transactions),
        round(avg_price_per_sqft, 2)
    ],
    'Formatted': [
        f'${total_sales_volume:,.0f}',
        f'${avg_sale_price:,.0f}',
        f'{total_transactions:,}',
        f'${avg_price_per_sqft:,.2f}'
    ]
})

print("=" * 50)
print("OVERALL KPI SUMMARY")
print("=" * 50)
for _, row in kpi_summary.iterrows():
    print(f"  {row['KPI']:30s} {row['Formatted']:>20s}")

OVERALL KPI SUMMARY
  Total Sales Volume                  $61,424,333,906
  Average Sale Price                       $1,066,376
  Total Transactions                           57,601
  Average Price Per SqFt                      $409.16


## 4 · Build Aggregated Datasets

In [7]:
# -- 7. Borough Summary
borough_summary = df.groupby('borough_name').agg(
    Total_Sales=('sale_price', 'sum'),
    Avg_Price=('sale_price', 'mean'),
    Median_Price=('sale_price', 'median'),
    Transactions=('sale_price', 'count'),
    Avg_Price_Per_SqFt=('price_per_sqft', 'mean'),
    Avg_Building_Age=('building_age', 'mean')
).reset_index()

borough_summary.columns = [
    'Borough', 'Total Sales', 'Avg Sale Price',
    'Median Sale Price', 'Transactions',
    'Avg Price Per SqFt', 'Avg Building Age'
]

borough_summary['Market Share Pct'] = (
    (borough_summary['Total Sales'] / borough_summary['Total Sales'].sum()) * 100
).round(2)

for col in ['Total Sales', 'Avg Sale Price', 'Median Sale Price', 'Avg Price Per SqFt', 'Avg Building Age']:
    borough_summary[col] = borough_summary[col].round(2)

print("Borough Summary:")
print(borough_summary.to_string(index=False))

Borough Summary:
      Borough       Total Sales  Avg Sale Price  Median Sale Price  Transactions  Avg Price Per SqFt  Avg Building Age  Market Share Pct
        Bronx  3,211,118,264.00      650,814.40         410,000.00          4934              250.42             72.77              5.23
     Brooklyn 16,277,171,774.00    1,078,672.75         775,000.00         15090              476.49             71.16             26.50
    Manhattan 27,369,919,174.00    1,980,313.96       1,126,624.00         13821              906.62             64.13             44.56
       Queens 11,520,434,172.00      642,451.16         500,000.00         17932              411.88             67.24             18.76
Staten Island  3,045,690,522.00      522,955.10         470,000.00          5824              324.27             46.53              4.96


In [8]:
# -- 8. Property Type Summary
property_summary = df.groupby('building_class_description').agg(
    Total_Sales=('sale_price', 'sum'),
    Avg_Price=('sale_price', 'mean'),
    Transactions=('sale_price', 'count'),
    Avg_Price_Per_SqFt=('price_per_sqft', 'mean')
).reset_index()

property_summary.columns = [
    'Property Type', 'Total Sales', 'Avg Sale Price',
    'Transactions', 'Avg Price Per SqFt'
]

for col in ['Total Sales', 'Avg Sale Price', 'Avg Price Per SqFt']:
    property_summary[col] = property_summary[col].round(2)

property_summary = property_summary.sort_values('Total Sales', ascending=False)

print(f"Property Type Summary ({len(property_summary)} types):")
print(property_summary.head(10).to_string(index=False))

Property Type Summary (44 types):
                 Property Type       Total Sales  Avg Sale Price  Transactions  Avg Price Per SqFt
  Condos - Elevator Apartments 18,206,161,845.00    1,798,139.44         10125                 NaN
   Coops - Elevator Apartments  8,665,289,025.00      756,793.80         11450               12.58
          One Family Dwellings  8,253,513,981.00      653,847.26         12623              420.68
          Two Family Dwellings  7,816,682,067.00      800,233.63          9768              379.46
   Rentals - Walkup Apartments  4,536,256,600.00    2,698,546.46          1681              410.57
        Three Family Dwellings  2,298,789,984.00      999,908.65          2299              353.44
Condos - 2-10 Unit Residential  1,477,366,588.00    1,435,730.41          1029                 NaN
               Store Buildings  1,276,312,133.00    2,811,260.20           454              606.04
     Coops - Walkup Apartments  1,191,667,019.00      478,773.41          2

In [9]:
# -- 9. Monthly Time Series
time_series = df.groupby(['sale_year', 'sale_month']).agg(
    Total_Sales=('sale_price', 'sum'),
    Avg_Price=('sale_price', 'mean'),
    Median_Price=('sale_price', 'median'),
    Transactions=('sale_price', 'count'),
    Avg_Price_Per_SqFt=('price_per_sqft', 'mean')
).reset_index()

time_series['Month Date'] = pd.to_datetime(
    time_series['sale_year'].astype(str) + '-' +
    time_series['sale_month'].astype(str).str.zfill(2) + '-01'
).dt.strftime('%Y-%m-%d')

time_series.columns = [
    'Sale Year', 'Sale Month', 'Total Sales', 'Avg Sale Price',
    'Median Sale Price', 'Transactions', 'Avg Price Per SqFt', 'Month Date'
]

for col in ['Total Sales', 'Avg Sale Price', 'Median Sale Price', 'Avg Price Per SqFt']:
    time_series[col] = time_series[col].round(2)

time_series = time_series.sort_values('Month Date')

col_order = ['Month Date', 'Sale Year', 'Sale Month', 'Total Sales',
             'Avg Sale Price', 'Median Sale Price', 'Transactions', 'Avg Price Per SqFt']
time_series = time_series[col_order]

print("Monthly Time Series:")
print(time_series.to_string(index=False))

Monthly Time Series:
Month Date  Sale Year  Sale Month      Total Sales  Avg Sale Price  Median Sale Price  Transactions  Avg Price Per SqFt
2016-09-01       2016           9 5,431,425,063.00    1,028,678.99         600,000.00          5280              391.45
2016-10-01       2016          10 4,325,206,915.00      979,218.23         610,000.00          4417              391.11
2016-11-01       2016          11 4,571,097,778.00      991,131.35         600,000.00          4612              388.54
2016-12-01       2016          12 5,649,526,882.00    1,088,540.83         616,520.50          5190              406.85
2017-01-01       2017           1 4,946,788,968.00    1,066,118.31         638,000.00          4640              403.16
2017-02-01       2017           2 4,479,339,993.00    1,047,553.79         620,000.00          4276              421.94
2017-03-01       2017           3 5,222,218,625.00    1,025,774.63         607,250.00          5091              401.86
2017-04-01       20

## 5 · Build Row-Level Dashboard Dataset

This is the primary Tableau data source. It keeps row-level granularity so that all interactive filters (borough, neighborhood, date range, property type, price bracket) work correctly. Columns are narrowed to only what a dashboard consumer needs.

In [10]:
# -- 10. Select and build final dashboard columns
final_dashboard = pd.DataFrame({
    'Sale Date': df['sale_date_str'],
    'Sale Year': df['sale_year'].astype('Int64'),
    'Sale Month': df['sale_month'].astype('Int64'),
    'Sale Quarter': df['sale_quarter'].astype('Int64'),
    'Day of Week': df['sale_day_of_week'],
    'Month Date': df['month_start_date'],
    'Borough': df['borough_name'],
    'Neighborhood': df['neighborhood'],
    'Zip Code': df['zip_code'],
    'Property Type': df['building_class_description'],
    'Tax Class': df['tax_class_at_time_of_sale'],
    'Residential Status': df['is_residential_flag'],
    'Total Units': df['total_units'],
    'Gross SqFt': df['gross_square_feet'],
    'Year Built': df['year_built'],
    'Building Age': df['building_age'],
    'Sale Price': df['sale_price'],
    'Price Per SqFt': df['price_per_sqft'],
    'Price Per Unit': df['price_per_unit'],
    'Size Category': df['size_category'],
    'Price Bracket': df['price_bracket']
})

print(f"Dashboard dataset: {final_dashboard.shape[0]:,} rows x {final_dashboard.shape[1]} columns")
print(f"\nColumns ({final_dashboard.shape[1]}):")
for i, col in enumerate(final_dashboard.columns, 1):
    dtype = final_dashboard[col].dtype
    nulls = final_dashboard[col].isnull().sum()
    print(f"  {i:2d}. {col:25s}  {str(dtype):12s}  nulls={nulls:>6,}")

Dashboard dataset: 57,601 rows x 21 columns

Columns (21):
   1. Sale Date                  object        nulls=     0
   2. Sale Year                  Int64         nulls=     0
   3. Sale Month                 Int64         nulls=     0
   4. Sale Quarter               Int64         nulls=     0


   5. Day of Week                object        nulls=     0
   6. Month Date                 object        nulls=     0
   7. Borough                    object        nulls=     0
   8. Neighborhood               object        nulls=     0


   9. Zip Code                   object        nulls=     0
  10. Property Type              object        nulls=     0
  11. Tax Class                  object        nulls=     0
  12. Residential Status         object        nulls=     0
  13. Total Units                int64         nulls=     0
  14. Gross SqFt                 float64       nulls=29,649
  15. Year Built                 Int64         nulls= 4,051
  16. Building Age               Int64         nulls= 4,051
  17. Sale Price                 float64       nulls=     0
  18. Price Per SqFt             float64       nulls=29,649
  19. Price Per Unit             float64       nulls=16,359
  20. Size Category              object        nulls=     0
  21. Price Bracket              object        nulls=     0


## 6 · Data Quality Validation

In [11]:
# -- 11. Validate critical fields and categorical cleanliness
print("=" * 55)
print("DATA QUALITY CHECKS")
print("=" * 55)

critical_fields = ['Borough', 'Sale Price', 'Sale Date', 'Property Type',
                   'Residential Status', 'Price Bracket']
print("\n[Critical Field Null Check]")
for field in critical_fields:
    missing = final_dashboard[field].isnull().sum()
    status = 'PASS' if missing == 0 else f'FAIL ({missing:,} missing)'
    print(f"  {field:25s} -> {status}")

print("\n[Categorical Value Counts]")
cat_cols = ['Borough', 'Residential Status', 'Size Category', 'Price Bracket', 'Day of Week']
for col in cat_cols:
    vals = final_dashboard[col].unique()
    print(f"  {col:25s} -> {len(vals)} values: {sorted(vals)[:8]}")

print("\n[Date Format Check]")
print(f"  Sale Date sample:   {final_dashboard['Sale Date'].iloc[0]}")
print(f"  Month Date sample:  {final_dashboard['Month Date'].iloc[0]}")

print("\n[Numeric Precision Check]")
print(f"  Price Per SqFt max decimals: {final_dashboard['Price Per SqFt'].dropna().apply(lambda x: len(str(x).split('.')[-1]) if '.' in str(x) else 0).max()}")
print(f"  Price Per Unit max decimals: {final_dashboard['Price Per Unit'].dropna().apply(lambda x: len(str(x).split('.')[-1]) if '.' in str(x) else 0).max()}")

print(f"\nFinal preview (transposed):")
print(final_dashboard.head(2).T)

DATA QUALITY CHECKS

[Critical Field Null Check]
  Borough                   -> PASS
  Sale Price                -> PASS
  Sale Date                 -> PASS
  Property Type             -> PASS
  Residential Status        -> PASS
  Price Bracket             -> PASS

[Categorical Value Counts]
  Borough                   -> 5 values: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
  Residential Status        -> 2 values: ['Non-Residential', 'Residential']
  Size Category             -> 6 values: ['Large', 'Medium', 'Mid-Large', 'Small', 'Unknown', 'Very Large']
  Price Bracket             -> 6 values: ['1M-2.5M', '2.5M-5M', '250K-500K', '500K-1M', '5M+', '<250K']
  Day of Week               -> 7 values: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

[Date Format Check]
  Sale Date sample:   2017-07-19
  Month Date sample:  2017-07-01

[Numeric Precision Check]
  Price Per SqFt max decimals: 2
  Price Per Unit max decimals: 2

Final preview (

## 7 · Export Final Datasets

In [12]:
# -- 12. Export: KPI Summary
kpi_path = os.path.join(OUTPUT_DIR, 'final_kpi_data.csv')
kpi_summary.to_csv(kpi_path, index=False)
print(f"Saved: {kpi_path} ({len(kpi_summary)} rows x {len(kpi_summary.columns)} cols)")
print(kpi_summary.to_string(index=False))

Saved: ../data/processed/final_kpi_data.csv (4 rows x 3 cols)
                   KPI             Value       Formatted
    Total Sales Volume 61,424,333,906.00 $61,424,333,906
    Average Sale Price      1,066,376.17      $1,066,376
    Total Transactions         57,601.00          57,601
Average Price Per SqFt            409.16         $409.16


In [13]:
# -- 13. Export: Time Series
ts_path = os.path.join(OUTPUT_DIR, 'final_time_series.csv')
time_series.to_csv(ts_path, index=False)
print(f"Saved: {ts_path} ({len(time_series)} rows x {len(time_series.columns)} cols)")

Saved: ../data/processed/final_time_series.csv (12 rows x 8 cols)


In [14]:
# -- 14. Export: Borough Summary
borough_path = os.path.join(OUTPUT_DIR, 'final_borough_summary.csv')
borough_summary.to_csv(borough_path, index=False)
print(f"Saved: {borough_path} ({len(borough_summary)} rows x {len(borough_summary.columns)} cols)")

Saved: ../data/processed/final_borough_summary.csv (5 rows x 8 cols)


In [15]:
# -- 15. Export: Property Type Summary
prop_path = os.path.join(OUTPUT_DIR, 'final_property_summary.csv')
property_summary.to_csv(prop_path, index=False)
print(f"Saved: {prop_path} ({len(property_summary)} rows x {len(property_summary.columns)} cols)")

Saved: ../data/processed/final_property_summary.csv (44 rows x 5 cols)


In [16]:
# -- 16. Export: Row-Level Dashboard Dataset
dashboard_path = os.path.join(OUTPUT_DIR, 'final_dashboard_data.csv')
final_dashboard.to_csv(dashboard_path, index=False)
print(f"Saved: {dashboard_path} ({len(final_dashboard):,} rows x {len(final_dashboard.columns)} cols)")

Saved: ../data/processed/final_dashboard_data.csv (57,601 rows x 21 cols)


## 8 · Export Summary

In [17]:
# -- 17. Final summary of all exports
print("=" * 70)
print("FINAL EXPORT SUMMARY")
print("=" * 70)

exports = [
    ('final_kpi_data.csv', len(kpi_summary), len(kpi_summary.columns),
     'KPI scorecard widgets'),
    ('final_time_series.csv', len(time_series), len(time_series.columns),
     'Monthly trend line and area charts'),
    ('final_borough_summary.csv', len(borough_summary), len(borough_summary.columns),
     'Borough-level comparison charts'),
    ('final_property_summary.csv', len(property_summary), len(property_summary.columns),
     'Property type breakdown charts'),
    ('final_dashboard_data.csv', len(final_dashboard), len(final_dashboard.columns),
     'Primary row-level source for interactive filters'),
]

for fname, rows, cols, purpose in exports:
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    unit = 'KB' if size_kb < 1024 else 'MB'
    size_display = size_kb if size_kb < 1024 else size_kb / 1024
    print(f"\n  {fname}")
    print(f"    Shape:   {rows:>8,} rows x {cols} cols")
    print(f"    Size:    {size_display:>8.1f} {unit}")
    print(f"    Purpose: {purpose}")

print("\n" + "=" * 70)
print("All datasets are ready for Tableau Public import.")
print("=" * 70)

FINAL EXPORT SUMMARY

  final_kpi_data.csv
    Shape:          4 rows x 3 cols
    Size:         0.2 KB
    Purpose: KPI scorecard widgets

  final_time_series.csv
    Shape:         12 rows x 8 cols
    Size:         0.8 KB
    Purpose: Monthly trend line and area charts

  final_borough_summary.csv
    Shape:          5 rows x 8 cols
    Size:         0.4 KB
    Purpose: Borough-level comparison charts

  final_property_summary.csv
    Shape:         44 rows x 5 cols
    Size:         2.4 KB
    Purpose: Property type breakdown charts

  final_dashboard_data.csv
    Shape:     57,601 rows x 21 cols
    Size:         8.7 MB
    Purpose: Primary row-level source for interactive filters

All datasets are ready for Tableau Public import.
